<a href="https://colab.research.google.com/github/ML-Bioinfo-CEITEC/bioinfo-school/blob/main/exercises/week3/B_protein_embeddings_esm2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Protein Embeddings with ESM-2


### Install & Import Libraries

In [ ]:
# Install necessary libraries
!pip install -qq transformers einops numpy pandas scikit-learn matplotlib seaborn umap-learn torch biopython

In [ ]:
import torch
from transformers import AutoTokenizer, EsmForMaskedLM, EsmModel
import einops
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
import umap.umap_ as umap
import matplotlib.pyplot as plt
import seaborn as sns

### Load ESM-2 Model and Tokenizer

In [ ]:
# Load pre-trained ESM-2 model and tokenizer
model_name = "facebook/esm2_t6_8M_UR50D" # Smallest model for demonstration
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmModel.from_pretrained(model_name)

## Embedding Extraction

### Load Protein Sequences

In [ ]:
from Bio import SeqIO

def load_fasta(fasta_file):
    sequences = []
    for record in SeqIO.parse(fasta_file, "fasta"):
        sequences.append(str(record.seq))
    return sequences

protein_sequences = load_fasta("/content/proteins.fasta")
print(f"Loaded {len(protein_sequences)} protein sequences.")
print("First sequence:", protein_sequences[0][:100], "...")

### Tokenize Sequences and Extract Embeddings

In [ ]:
def get_embeddings(sequences, tokenizer, model):
    all_per_residue_embeddings = []
    all_per_sequence_embeddings = []

    for seq in sequences:
        inputs = tokenizer(seq, return_tensors='pt', truncation=True, max_length=1024)
        with torch.no_grad():
            outputs = model(**inputs)

        # Per-residue embeddings (all hidden states)
        # The last hidden state is typically used for per-residue embeddings
        per_residue_embeddings = outputs.last_hidden_state.squeeze(0).cpu().numpy()
        all_per_residue_embeddings.append(per_residue_embeddings)

        # Per-sequence embedding (CLS token embedding)
        # The first token (CLS token) is typically used for sequence-level representation
        per_sequence_embedding = outputs.last_hidden_state[:, 0].squeeze(0).cpu().numpy()
        all_per_sequence_embeddings.append(per_sequence_embedding)

    return all_per_residue_embeddings, all_per_sequence_embeddings

per_residue_embeddings, per_sequence_embeddings = get_embeddings(protein_sequences, tokenizer, model)

print(f"Extracted {len(per_residue_embeddings)} sets of per-residue embeddings.")
print(f"Shape of first per-residue embedding: {per_residue_embeddings[0].shape}")
print(f"Extracted {len(per_sequence_embeddings)} per-sequence embeddings.")
print(f"Shape of first per-sequence embedding: {per_sequence_embeddings[0].shape}")

## Dimensionality Reduction and Clustering

### Load Protein Accessions (Metadata)

In [ ]:
import pandas as pd
# Load protein accessions to link with embeddings for better analysis
protein_accessions_df = pd.read_csv("/content/protein_accessions.tsv", sep="\t")
print(f"Loaded {len(protein_accessions_df)} protein accessions.")
print("First 5 accessions:\n", protein_accessions_df.head())

### UMAP Dimensionality Reduction

In [ ]:
import numpy as np
import umap.umap_ as umap

# Convert list of arrays to a single 2D NumPy array
per_sequence_embeddings_array = np.array(per_sequence_embeddings)

# UMAP
umapper = umap.UMAP(random_state=42)
umap_embeddings = umapper.fit_transform(per_sequence_embeddings_array)

print(f"UMAP embeddings shape: {umap_embeddings.shape}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Add UMAP embeddings to the DataFrame for plotting
protein_accessions_df['umap_x'] = umap_embeddings[:, 0]
protein_accessions_df['umap_y'] = umap_embeddings[:, 1]

# Rename the '# family' column to 'family' for easier plotting
if '# family' in protein_accessions_df.columns:
    protein_accessions_df = protein_accessions_df.rename(columns={'# family': 'family'})

plt.figure(figsize=(10, 8))
sns.scatterplot(x='umap_x', y='umap_y', hue='family', data=protein_accessions_df, palette='tab10', s=70) # Changed palette to 'tab10'
plt.title('UMAP of Protein Embeddings by Family')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.legend(title='Protein Family', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()